# **Group 2 - HW #6 - Chapter 6 - Adrian Angeles**

## Q1: Most Profitable Items

- Joins Sales.SalesOrderDetail sod and Production.P and Sales.SalesOrderHeader
- Uses Except to remove any products sold below their standard cost.
- Could be used to find the most popular or highest-grossing items among the total inventory for marketing and sales purposes

In [10]:
SELECT TOP (10) p.ProductID, p.Name,
       SUM(sod.OrderQty*sod.UnitPrice*(1-sod.UnitPriceDiscount)) AS Revenue
FROM Production.Product p
JOIN Sales.SalesOrderDetail sod ON sod.ProductID=p.ProductID
JOIN Sales.SalesOrderHeader soh ON soh.SalesOrderID=sod.SalesOrderID
WHERE p.ProductID IN (
  SELECT DISTINCT ProductID FROM Sales.SalesOrderDetail
  EXCEPT
  SELECT sod.ProductID FROM Sales.SalesOrderDetail sod
  JOIN Production.Product q ON q.ProductID=sod.ProductID
  WHERE sod.UnitPrice*(1-sod.UnitPriceDiscount) < q.StandardCost )
GROUP BY p.ProductID,p.Name
ORDER BY Revenue DESC;


(10 rows affected)

Total execution time: 00:00:02.722

ProductID,Name,Revenue
786,"Mountain-300 Black, 40",501648.8751
787,"Mountain-300 Black, 44",484051.518
788,"Mountain-300 Black, 48",479071.9001
785,"Mountain-300 Black, 38",442477.087
981,"Mountain-400-W Silver, 40",323703.8207
980,"Mountain-400-W Silver, 38",241773.758
876,Hitch Rack - 4-Bike,237096.156
983,"Mountain-400-W Silver, 46",227347.6673
982,"Mountain-400-W Silver, 42",217457.874
832,"ML Mountain Frame - Black, 48",200284.4978


## <span style="color: var(--vscode-foreground);">Q2: Commision Estimate</span>

- <span style="color: var(--vscode-foreground);">Computes the estimated comission through&nbsp;</span>  SUM<span style="color: rgb(33, 33, 33); font-family: Consolas, &quot;Courier New&quot;, monospace; font-size: 12px; white-space: pre;">(</span><span style="font-family: Consolas, &quot;Courier New&quot;, monospace; font-size: 12px; white-space: pre; color: rgb(9, 136, 90);">0</span><span style="color: rgb(33, 33, 33); font-family: Consolas, &quot;Courier New&quot;, monospace; font-size: 12px; white-space: pre;">.</span><span style="font-family: Consolas, &quot;Courier New&quot;, monospace; font-size: 12px; white-space: pre; color: rgb(9, 136, 90);">05</span>_<span style="color: rgb(33, 33, 33); font-family: Consolas, &quot;Courier New&quot;, monospace; font-size: 12px; white-space: pre;">sod.UnitPrice</span>_<span style="color: rgb(33, 33, 33); font-family: Consolas, &quot;Courier New&quot;, monospace; font-size: 12px; white-space: pre;">(</span><span style="font-family: Consolas, &quot;Courier New&quot;, monospace; font-size: 12px; white-space: pre; color: rgb(9, 136, 90);">1</span><span style="font-family: Consolas, &quot;Courier New&quot;, monospace; font-size: 12px; white-space: pre; color: rgb(0, 0, 0);">-</span><span style="color: rgb(33, 33, 33); font-family: Consolas, &quot;Courier New&quot;, monospace; font-size: 12px; white-space: pre;">sod.UnitPriceDiscount)</span><span style="font-family: Consolas, &quot;Courier New&quot;, monospace; font-size: 12px; white-space: pre; color: rgb(0, 0, 0);">*</span><span style="color: rgb(33, 33, 33); font-family: Consolas, &quot;Courier New&quot;, monospace; font-size: 12px; white-space: pre;">sod.OrderQty)</span>
- <span style="color: var(--vscode-foreground);">Used to display Estimated Online vs Offline Comissions by Highest Commisions to Lowest using Union All</span>
- <span style="color: var(--vscode-foreground);">Can be used to predict total earnings per sale for documentation, tax purposes, or for tracking general revenue</span>

In [7]:
SELECT 'Online' AS Channel,
       SUM(0.05*sod.UnitPrice*(1-sod.UnitPriceDiscount)*sod.OrderQty) AS EstCommission
FROM Sales.SalesOrderDetail sod JOIN Sales.SalesOrderHeader soh ON soh.SalesOrderID=sod.SalesOrderID
WHERE soh.OnlineOrderFlag=1
UNION ALL
SELECT 'Offline',
       SUM(0.05*sod.UnitPrice*(1-sod.UnitPriceDiscount)*sod.OrderQty)
FROM Sales.SalesOrderDetail sod JOIN Sales.SalesOrderHeader soh ON soh.SalesOrderID=sod.SalesOrderID
WHERE soh.OnlineOrderFlag=0;


(2 rows affected)

Total execution time: 00:00:00.181

Channel,EstCommission
Online,1467933.861035
Offline,4024385.214242


## Q3:  Slow Shipment List (Takes more than a week to ship)

- Uses Sales.SalesOrderHeader to find total timeline
- Filtered by Shipped Orders who would take more than 7 days to ship using Except
- Could be used to determine where there could be improvement in shipment areas or methods of delivery

In [15]:
SELECT SalesOrderID, OrderDate, ShipDate
FROM Sales.SalesOrderHeader
WHERE ShipDate IS NOT NULL
EXCEPT
SELECT SalesOrderID, OrderDate, ShipDate
FROM Sales.SalesOrderHeader
WHERE ShipDate IS NOT NULL AND DATEDIFF(DAY,OrderDate,ShipDate) <= 7;

(9 rows affected)

Total execution time: 00:00:00.069

SalesOrderID,OrderDate,ShipDate
65089,2014-01-28 00:00:00.000,2014-02-05 00:00:00.000
65090,2014-01-28 00:00:00.000,2014-02-05 00:00:00.000
67202,2014-02-28 00:00:00.000,2014-03-08 00:00:00.000
67203,2014-02-28 00:00:00.000,2014-03-08 00:00:00.000
67204,2014-02-28 00:00:00.000,2014-03-08 00:00:00.000
69309,2014-03-30 00:00:00.000,2014-04-07 00:00:00.000
69310,2014-03-30 00:00:00.000,2014-04-07 00:00:00.000
71691,2014-04-30 00:00:00.000,2014-05-08 00:00:00.000
71692,2014-04-30 00:00:00.000,2014-05-08 00:00:00.000


## Q4: Sales at a Loss

- Joins Sales.SalesOrderDetail sod and Production.product p
- Calculates Net Unit Price through \[sod.UnitPrice\*(1-sod.UnitPriceDiscount)\]
- Compares the Standard Cost by the Unit Price to find Unit Margin
- Could be used to determine where the biggest profit losses are and take appropriate actions towards them

In [14]:
SELECT sod.SalesOrderID, sod.ProductID,
       (sod.UnitPrice*(1 - sod.UnitPriceDiscount)) AS NetUnitPrice,
       p.StandardCost,
       (sod.UnitPrice*(1 - sod.UnitPriceDiscount) - p.StandardCost) AS UnitMargin
FROM Sales.SalesOrderDetail sod
JOIN Production.Product p ON p.ProductID = sod.ProductID
WHERE (sod.UnitPrice*(1 - sod.UnitPriceDiscount)) < p.StandardCost
ORDER BY UnitMargin ASC;


(29161 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.294

SalesOrderID,ProductID,NetUnitPrice,StandardCost,UnitMargin
46323,771,552.4984,1912.1544,-1359.656
46323,772,552.4984,1912.1544,-1359.656
46323,773,552.4984,1912.1544,-1359.656
46323,774,552.4984,1912.1544,-1359.656
46327,774,552.4984,1912.1544,-1359.656
46327,771,552.4984,1912.1544,-1359.656
46330,772,552.4984,1912.1544,-1359.656
46330,771,552.4984,1912.1544,-1359.656
46330,773,552.4984,1912.1544,-1359.656
46332,772,552.4984,1912.1544,-1359.656


## Q5: Top Products by Discount Dollars (Offline and Online Sales)

- Calculates total discount amounts per product within the last 90 days and seperates Offline and Online Sales using Union All.
- Uses MaxOrderDate to find latest order date and Filter to the last 90 days
- Shows which products have the largest total discount amounts in price given to customers, can be used for tracking discount amounts and performance

In [38]:
DECLARE @MaxOrderDate date = (SELECT MAX(OrderDate) FROM Sales.SalesOrderHeader);
SELECT ProductID, SUM(DiscountDollars) AS DiscountDollars_90D
FROM (
  SELECT sod.ProductID, sod.UnitPrice*sod.UnitPriceDiscount*sod.OrderQty AS DiscountDollars
  FROM Sales.SalesOrderDetail sod JOIN Sales.SalesOrderHeader soh ON soh.SalesOrderID=sod.SalesOrderID
  WHERE soh.OrderDate>=DATEADD(DAY,-90,@MaxOrderDate)
  UNION ALL
  SELECT sod.ProductID, sod.UnitPrice*sod.UnitPriceDiscount*sod.OrderQty
  FROM Sales.SalesOrderDetail sod JOIN Sales.SalesOrderHeader soh ON soh.SalesOrderID=sod.SalesOrderID
  WHERE soh.OrderDate>=DATEADD(DAY,-90,@MaxOrderDate)
) x
GROUP BY ProductID
ORDER BY DiscountDollars_90D DESC;

(171 rows affected)

Total execution time: 00:00:00.152

ProductID,DiscountDollars_90D
976,2599.7935
988,2169.5616
985,1807.968
987,1762.7688
984,1355.976
986,994.3824
957,331.8624
864,304.287
998,267.2946
974,256.5095


## Q6: Top Customers by Total Sales Made

- <span style="color: var(--vscode-foreground);">Identifies top 20 customers by revenue for two consecutive years</span>
- <span style="color: var(--vscode-foreground);">Total Sum per customer found through SUM(TotalDue), Sorts by Biggest Lifetime Earners</span>
- Could be used to filter through the customerbase and find the biggest spenders, giving priority to them or offering perks or rewards

In [37]:
SELECT TOP (20) CustomerID, SUM(TotalDue) AS LifetimeValue
FROM (
  SELECT CustomerID, TotalDue FROM Sales.SalesOrderHeader WHERE OnlineOrderFlag=1
  UNION ALL
  SELECT CustomerID, TotalDue FROM Sales.SalesOrderHeader WHERE OnlineOrderFlag=0
) x
GROUP BY CustomerID
ORDER BY LifetimeValue DESC;

(20 rows affected)

Total execution time: 00:00:00.032

CustomerID,LifetimeValue
29818,989184.082
29715,961675.8596
29722,954021.9235
30117,919801.8188
29614,901346.856
29639,887090.4106
29701,841866.5522
29617,834475.9271
29994,824331.7682
29646,820383.5466


## Q7: Biggest Repeat Buyers

- <span style="color: var(--vscode-foreground);">Counts customer orders for the first and second halves of the year and uses Union to combine customers from either period</span>
- <span style="color: var(--vscode-foreground);">Sorted by CustomerID's and Total Individual Sales Made, from Biggest to Lowest</span>
- Could be used to determine which groups or individuals rank highest in loyalty, focusing trends or offers towards them

In [36]:
DECLARE @MaxOrderDate date = (SELECT MAX(OrderDate) FROM Sales.SalesOrderHeader);
SELECT CustomerID, COUNT(*) AS Orders_6M
FROM (
  SELECT CustomerID FROM Sales.SalesOrderHeader WHERE OrderDate>=DATEADD(MONTH,-6,@MaxOrderDate) AND OnlineOrderFlag=1
  UNION ALL
  SELECT CustomerID FROM Sales.SalesOrderHeader WHERE OrderDate>=DATEADD(MONTH,-6,@MaxOrderDate) AND OnlineOrderFlag=0
) x
GROUP BY CustomerID HAVING COUNT(*)>=3
ORDER BY Orders_6M DESC;

(135 rows affected)

Total execution time: 00:00:00.042

CustomerID,Orders_6M
11185,19
11276,18
11262,16
11711,16
11300,15
11176,13
11277,13
11091,13
11223,12
11200,12


## Q8: Revenue Per Area

- Calculates revnue from each territory over the past year and combines both with Union All
- Categorizes Region and Total Area Revenue (soh.TotalDue, st.Name)
- Could be used to find which Areas are the most profitable, shifting focus and planning priorities

In [33]:
DECLARE @MaxOrderDate date = (SELECT MAX(OrderDate) FROM Sales.SalesOrderHeader);
SELECT Name AS Territory, SUM(Revenue) AS Revenue_12M
FROM (
  SELECT st.Name, soh.TotalDue AS Revenue
  FROM Sales.SalesOrderHeader soh JOIN Sales.SalesTerritory st ON st.TerritoryID=soh.TerritoryID
  WHERE soh.OrderDate>=DATEADD(DAY,-365,@MaxOrderDate) AND soh.OnlineOrderFlag=1
  UNION ALL
  SELECT st.Name, soh.TotalDue
  FROM Sales.SalesOrderHeader soh JOIN Sales.SalesTerritory st ON st.TerritoryID=soh.TerritoryID
  WHERE soh.OrderDate>=DATEADD(DAY,-365,@MaxOrderDate) AND soh.OnlineOrderFlag=0
) x
GROUP BY Name
ORDER BY Revenue_12M DESC;

(10 rows affected)

Total execution time: 00:00:00.052

Territory,Revenue_12M
Southwest,10752323.337
Northwest,8327448.4745
Canada,7014746.4606
Australia,6319248.0307
United Kingdom,5294241.7085
France,5179240.8931
Germany,3928538.5583
Central,3218889.5773
Southeast,2661896.342
Northeast,2492475.942


## <span style="color: var(--vscode-foreground);">Q9: Average Discount Percent (Last 180 Days)</span>

- <span style="color: var(--vscode-foreground);">Finds products averaging at least a 10% discount over the last 180 days</span>
- Uses a second query to check which products still sold above cost and combines them with Union All.
- <span style="color: var(--vscode-foreground);">Calculates the average discount amount given per item, Can be used to help balance where discounts should be placed in tandem with performance of said item</span>

In [31]:
DECLARE @MaxOrderDate date = (SELECT MAX(OrderDate) FROM Sales.SalesOrderHeader);
SELECT ProductID, AVG(AvgDiscount) AS AvgDiscountPct_180D
FROM (
  SELECT sod.ProductID, 100.0*sod.UnitPriceDiscount AS AvgDiscount
  FROM Sales.SalesOrderDetail sod JOIN Sales.SalesOrderHeader soh ON soh.SalesOrderID=sod.SalesOrderID
  WHERE soh.OrderDate>=DATEADD(DAY,-180,@MaxOrderDate) AND soh.OnlineOrderFlag=1
  UNION ALL
  SELECT sod.ProductID, 100.0*sod.UnitPriceDiscount
  FROM Sales.SalesOrderDetail sod JOIN Sales.SalesOrderHeader soh ON soh.SalesOrderID=sod.SalesOrderID
  WHERE soh.OrderDate>=DATEADD(DAY,-180,@MaxOrderDate) AND soh.OnlineOrderFlag=0
) x
GROUP BY ProductID
ORDER BY AvgDiscountPct_180D DESC;

(180 rows affected)

Total execution time: 00:00:00.110

ProductID,AvgDiscountPct_180D
988,18.536585
985,18.500000
987,18.461538
986,16.615384
984,15.308641
864,0.600000
867,0.398523
869,0.349673
875,0.280991
884,0.232558


## Q10: Product With The Most Order Lines (From The Last 90 Days)

- <span style="color: var(--vscode-foreground);">Counts how many order lines each product appeared in during the last 90 days.</span>
- <span style="color: var(--vscode-foreground);">Separates online and offline order data and uses Union All to combine both sets</span>
- Counts how frequently an item has been ordered in the last 90 says, can be used to track trends in sales and evaluate which items are in demand

In [1]:
DECLARE @MaxOrderDate date = (SELECT MAX(OrderDate) FROM Sales.SalesOrderHeader);
SELECT ProductID, COUNT(*) AS Lines_90D
FROM (
  SELECT sod.ProductID
  FROM Sales.SalesOrderDetail sod JOIN Sales.SalesOrderHeader soh ON soh.SalesOrderID=sod.SalesOrderID
  WHERE soh.OrderDate>=DATEADD(DAY,-90,@MaxOrderDate) AND soh.OnlineOrderFlag=1
  UNION ALL
  SELECT sod.ProductID
  FROM Sales.SalesOrderDetail sod JOIN Sales.SalesOrderHeader soh ON soh.SalesOrderID=sod.SalesOrderID
  WHERE soh.OrderDate>=DATEADD(DAY,-90,@MaxOrderDate) AND soh.OnlineOrderFlag=0
) x
GROUP BY ProductID
ORDER BY Lines_90D DESC;

(171 rows affected)

Total execution time: 00:00:00.101

ProductID,Lines_90D
870,1008
873,748
921,730
922,582
711,578
712,567
707,567
878,528
708,526
871,478
